In [0]:
import re
import logging
from datetime import datetime
from pyspark.sql import SparkSession
import pandas as pd

from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType, TimestampType
)

from setup import load_config, get_all_tables,get_raw_path
from src.utils.schemas import ORDERS_SCHEMA,CUSTOMER_SCHEMA,PRODUCTS_SCHEMA
import pyspark.sql.functions as F

config = load_config()

# unpack into a dict — same names as your original constants
tables = get_all_tables(config)
path=get_raw_path(config)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("Ingestion layer")

 
# ── Volume path for raw uploaded files ───────────────────────────────────────
# Files are uploaded via UI to: Catalog > retail_catalog > retail > raw_files
VOLUME_RAW = path
 
RAW_PRODUCTS  = f"{VOLUME_RAW}/Products.csv"
RAW_CUSTOMERS = f"{VOLUME_RAW}/Customer.xlsx"
RAW_ORDERS    = f"{VOLUME_RAW}/Orders.json"

BRONZE_PRODUCTS  = tables['BRONZE_PRODUCTS']
BRONZE_CUSTOMERS = tables['BRONZE_CUSTOMERS']
BRONZE_ORDERS    = tables['BRONZE_ORDERS']
 
SILVER_PRODUCTS  = tables['SILVER_PRODUCTS']
SILVER_CUSTOMERS = tables['SILVER_CUSTOMERS']
SILVER_ORDERS    = tables['SILVER_ORDERS']
 
GOLD_SALES       = tables["GOLD_SALES"]
RUN_ID       = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
INGESTION_TS = datetime.utcnow().isoformat()

In [0]:
def add_audit_columns(df, source: str):
    """Stamp every Bronze row with source file, ingest time, and run ID."""
    return (
        df
        .withColumn("_source",       F.lit(source))
        .withColumn("_ingested_at",  F.lit(INGESTION_TS).cast(TimestampType()))
        .withColumn("_pipeline_run", F.lit(RUN_ID))
    )
 

In [0]:
def write_table(df, table: str, mode: str = "overwrite", partition_by=None):
    """Write DataFrame as a Unity Catalog managed Delta table."""
    writer = df.write.format("delta").mode(mode).option("overwriteSchema", "true")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.saveAsTable(table)
    n = spark.table(table).count()
    logger.info(f"Written {n:,} rows → {table}  (mode={mode})")
    return n

In [0]:
def optimize_table(table: str, zorder_cols: list = None):
    sql = f"OPTIMIZE {table}"
    if zorder_cols:
        sql += f" ZORDER BY ({', '.join(zorder_cols)})"
    spark.sql(sql)
    logger.info(f"  OPTIMIZE complete: {table}")

In [0]:

logger.info("BRONZE | Loading Products.csv …")

try:
    df_products_bronze = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "false")
             .schema(PRODUCTS_SCHEMA)
             .csv(RAW_PRODUCTS)
    )
    df_products_bronze = add_audit_columns(df_products_bronze, "Products.csv")
    rows_in = df_products_bronze.count()
    # Full overwrite — products is a small, slowly-changing dimension#
    write_table(df_products_bronze, BRONZE_PRODUCTS, mode="overwrite")
    optimize_table(BRONZE_PRODUCTS, zorder_cols=["product_id"])
except Exception as e:
    logger.error(f"Failed to load Products.csv: {e}")
    raise

In [0]:
logger.info("BRONZE | Loading Customer.xlsx via pandas …")

try:

    pandas_customers = pd.read_excel(
        RAW_CUSTOMERS,       # Volume paths work directly with pandas
        dtype=str,           # keep all columns as text at Bronze; cast in Silver
        engine="openpyxl"
    )
    pandas_customers.columns = [
        c.strip()
         .replace(' ', '_')
         .replace(',', '')
         .replace(';', '')
         .replace('{', '')
         .replace('}', '')
         .replace('(', '')
         .replace(')', '')
         .replace('\n', '')
         .replace('\t', '')
         .replace('=', '')
        for c in pandas_customers.columns
    ]

    df_customers_bronze = spark.createDataFrame(pandas_customers)
    df_customers_bronze = add_audit_columns(df_customers_bronze, "Customer.xlsx")
    rows_in = df_customers_bronze.count()

    write_table(df_customers_bronze, BRONZE_CUSTOMERS, mode="overwrite")
    optimize_table(BRONZE_CUSTOMERS, zorder_cols=["customer_id"])
except Exception as e:
    logger.error(f"Failed to load Customer.xlsx: {e}")
    raise

In [0]:
 
logger.info("BRONZE | Loading Orders.json …")

try:
    df_orders_bronze = (
        spark.read
             .option("multiline", "true")
             .json(RAW_ORDERS)
    )
    df_orders_bronze = df_orders_bronze.select(
        col("Row ID").cast(IntegerType()).alias("row_id"),
        col("Order ID").cast(StringType()).alias("order_id"),
        col("Order Date").cast(StringType()).alias("order_date"),
        col("Ship Date").cast(StringType()).alias("ship_date"),
        col("Ship Mode").cast(StringType()).alias("ship_mode"),
        col("Customer ID").cast(StringType()).alias("customer_id"),
        col("Product ID").cast(StringType()).alias("product_id"),
        col("Quantity").cast(IntegerType()).alias("quantity"),
        col("Price").cast(DoubleType()).alias("price"),
        col("Discount").cast(DoubleType()).alias("discount"),
        col("Profit").cast(DoubleType()).alias("profit")
    )
    df_orders_bronze = add_audit_columns(df_orders_bronze, "Orders.json")
    rows_in = df_orders_bronze.count()
    write_table(df_orders_bronze, BRONZE_ORDERS, mode="overwrite")
    optimize_table(BRONZE_ORDERS, zorder_cols=["customer_id", "product_id"])
except Exception as e:
    logger.error(f"Failed to load Orders.json: {e}")
    raise